# Index Management — End-to-End Demo

This notebook walks through all four workflows in the `index_management` package:

1. **Universe** — resolve ISIN codes to ticker symbols
2. **Market** — fetch prices and market caps from Yahoo Finance
3. **Strategy** — compute portfolio weights (Cap-Weighted & Max Sharpe Ratio)
4. **Valuation** — convert weights + prices into share counts and portfolio value

Each section corresponds to its detailed driver notebook in `index_management/drivers/`.

In [ ]:
TARGET_DATE = "20250331"  # change this to run a different quarter

## 1 · Universe

Maps ISIN codes in `data/universe/raw/` to Yahoo Finance ticker symbols and writes `data/universe/processed/`.

In [ ]:
from index_management.universe.Manager import Universe
from index_management.utilities.utils import fullpath, get_datestr
import pandas as pd

unv = Universe(TARGET_DATE)

# Read the already-processed universe (run get_raw_universe() to refresh from source)
unv_path = fullpath(unv.path_universe("processed"), get_datestr(TARGET_DATE) + ".csv")
df_universe = pd.read_csv(unv_path)
print(f"{len(df_universe)} securities in universe")
df_universe.head()

## 2 · Market

Fetches closing prices and market caps from Yahoo Finance for all universe members.

In [ ]:
from index_management.market.Manager import Market

mkt = Market(TARGET_DATE)

# Read cached market data (call mkt.get_prices() / mkt.get_caps() to refresh)
import pandas as pd
from index_management.utilities.utils import fullpath, get_datestr

prices = pd.read_csv(fullpath("data", "market", "prices", get_datestr(TARGET_DATE) + ".csv"), index_col=0)
caps   = pd.read_csv(fullpath("data", "market", "caps",   get_datestr(TARGET_DATE) + ".csv"))

print(f"Price history shape: {prices.shape}")
print(f"Market caps: {len(caps)} symbols")
caps.head()

## 3 · Strategy

Computes portfolio weights using two methods:
- **Cap-Weighted (`cw`)** — proportional to market cap
- **Max Sharpe Ratio (`msr`)** — mean-variance optimisation with Ledoit-Wolf covariance shrinkage

In [ ]:
from index_management.strategy.Manager import CapWeight

cw = CapWeight(TARGET_DATE)
prior_cw, df_cw = cw.calculate_weights()
print(f"Cap-weight sum: {df_cw['Weights_new'].sum():.6f}")
df_cw.head()

In [ ]:
from index_management.strategy.Manager import MaxSharpeRatioPortfolio
import matplotlib.pyplot as plt
import seaborn as sns

msr = MaxSharpeRatioPortfolio(TARGET_DATE)
prior_msr, df_msr = msr.calculate_weights()

df_msr.set_index("Asset", inplace=True)
df_msr.plot(kind="bar", figsize=(12, 5), title="MSR Weights — Sample vs Ledoit-Wolf")
plt.ylabel("Weight")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(msr.df_sample_cov_matrix, annot=False, cmap="coolwarm", ax=axes[0]).set_title("Sample Covariance")
sns.heatmap(msr.df_LedoitWolf_cov_matrix, annot=False, cmap="coolwarm", ax=axes[1]).set_title("Ledoit-Wolf Covariance")
plt.tight_layout()
plt.show()

## 4 · Valuation

Converts weights and end-of-quarter prices into a share allocation for a $1B notional portfolio.

In [ ]:
from index_management.valuation.Manager import Valuation

val_cw = Valuation(TARGET_DATE, "cw")
df_alloc_cw = val_cw.valuation_quarterly()
print("Cap-weight quarterly allocation:")
df_alloc_cw.head(10)

In [ ]:
val_msr = Valuation(TARGET_DATE, "msr")
df_alloc_msr = val_msr.valuation_quarterly()
print("MSR quarterly allocation:")
df_alloc_msr.head(10)